<a href="https://colab.research.google.com/github/sarafawzyb/travel-app/blob/master/notebooks/Getting_started_with_google_colab_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# 1. Import Libraries
# ================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
# ================================
# 2. Load Dataset
# ================================
columns = [
    "age","workclass","fnlwgt","education","education_num",
    "marital_status","occupation","relationship","race","sex",
    "capital_gain","capital_loss","hours_per_week","native_country","income"
]

df = pd.read_csv(
    "adult.data",
    names=columns,
    sep=",",
    skipinitialspace=True
)

In [ ]:
# ================================
# 3. Handle Missing Values
# ================================
df.replace("?", np.nan, inplace=True)

for col in df.select_dtypes(include="object"):
    df[col].fillna(df[col].mode()[0], inplace=True)

In [ ]:
# ================================
# 4. Encode Target Variable
# ================================
df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

In [ ]:
# ================================
# 5. Feature / Target Split
# ================================
X = df.drop("income", axis=1)
y = df["income"]

# ================================
# 6. Numerical & Categorical Columns
# ================================
num_features = [
    "age", "fnlwgt", "education_num",
    "capital_gain", "capital_loss", "hours_per_week"
]

cat_features = [
    "workclass","education","marital_status","occupation",
    "relationship","race","sex","native_country"
]

In [ ]:
# ================================
# 7. Preprocessing Pipeline
# ================================
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

# ================================
# 8. Train-Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# ================================
# 9. Visualization (EDA)
# ================================
# Class Distribution
sns.countplot(x=y)
plt.title("Income Class Distribution")
plt.show()

# Age Distribution
sns.histplot(df["age"], bins=30)
plt.title("Age Distribution")
plt.show()

# Correlation Heatmap
plt.figure(figsize=(8,6))
sns.heatmap(df[num_features + ["income"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ================================
# 10. Models Dictionary
# ================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [ ]:
# ================================
# 11. Training & Evaluation
# ================================
results = []

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append([name, acc, prec, rec, f1])

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix – {name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

In [ ]:
# ================================
# 12. Results DataFrame
# ================================
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-score"]
)

print(results_df)

In [ ]:
# ================================
# 13. Model Comparison Visualization
# ================================
results_df.set_index("Model")[["Accuracy","Precision","Recall","F1-score"]].plot(
    kind="bar", figsize=(12,6)
)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.show()